# AMD 三份“季度收入创纪录”公告分母修复

对准确定位的三份原始 SEC 原件，只读取明确本季收入句，保留其余 123 事件数值和两个 AMD 定性缺口。

In [1]:
import json,hashlib,re
from pathlib import Path
from bs4 import BeautifulSoup
root=Path.cwd();card=root/'research/stage1_amd_nvda_guidance_breadth_source_20261023';raw=root/'data/snapshots/stage1_amd_nvda_sec_guidance_original_probe_20261023';plan=json.loads((card/'numeric_outlier_repair_plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and sha(card/'plan.json')==plan['original_plan_sha256'] and sha(card/'numeric_repair_result.json')==plan['attempt2_result_sha256'] and not (card/'numeric_outlier_repair_result.json').exists()
prior=json.loads((card/'numeric_repair_result.json').read_text());rows=prior['rows'];by={x['accession']:x for x in rows};assert len(rows)==126 and set(plan['exact_accessions'])=={x['accession'] for x in prior['large_source_ratio_review_rows']}
changes=[]
for acc in plan['exact_accessions']:
 row=by[acc];p=raw/f'item202_{acc}.txt';assert sha(p)==row['raw_sha256'];body=p.read_bytes();parts=[]
 for doc in re.findall(rb'<DOCUMENT>(.*?)</DOCUMENT>',body,re.S|re.I):
  typ=re.search(rb'<TYPE>\s*([^\r\n<]+)',doc,re.I);content=re.search(rb'<TEXT>(.*?)</TEXT>',doc,re.S|re.I)
  if typ and content and typ.group(1).strip().upper()==b'EX-99.1':parts.append(re.sub(r'\s+',' ',BeautifulSoup(content.group(1).decode('utf8','replace'),'html.parser').get_text(' ',strip=True)))
 text=' '.join(parts);m=re.search(r'\b(First|Second|Third|Fourth)\s+quarter\s+revenue\s+was\s+a\s+record\s+\$\s*([\d,.]+)\s*billion',text,re.I);assert m,(acc,'no unique quarter prose');reported=float(m.group(2).replace(',',''))*1000;forecast=row['forecast_next_mid_USD_million'];assert forecast and reported>0;growth=forecast/reported-1;assert -1<growth<1.5
 changes.append({'accession':acc,'filing_date':row['filing_date'],'old_reported_USD_million':row['reported_current_USD_million'],'new_reported_USD_million':reported,'unchanged_forward_mid_USD_million':forecast,'new_growth':growth,'original_quarter_sentence':m.group(0)})
 row.update(reported_current_USD_million=reported,reported_current_excerpt=m.group(0),growth=growth,reason='absolute_guidance_over_explicit_current_quarter_record_revenue')
unknown=[{'issuer':x['issuer'],'filing_date':x['filing_date'],'accession':x['accession'],'reason':x['reason'],'guidance_excerpt':x['guidance_excerpt']} for x in rows if x['status']!='numeric'];assert len(unknown)==2 and all(x['issuer']=='AMD' for x in unknown)
large=[{'issuer':x['issuer'],'filing_date':x['filing_date'],'accession':x['accession'],'growth':x['growth']} for x in rows if x['growth'] is not None and abs(x['growth'])>0.6];assert not large
counts={issuer:{'numeric':sum(x['issuer']==issuer and x['status']=='numeric' for x in rows),'qualitative_unknown':sum(x['issuer']==issuer and x['status']!='numeric' for x in rows)} for issuer in ('AMD','NVDA')};assert counts=={'AMD':{'numeric':61,'qualitative_unknown':2},'NVDA':{'numeric':63,'qualitative_unknown':0}}
result={'status':'source_numeric_124_unambiguous_and_two_AMD_qualitative_unknown_no_market_outcomes','plan_sha256':sha(card/'numeric_outlier_repair_plan.json'),'parent_attempt2_sha256':plan['attempt2_result_sha256'],'rows':rows,'changes':changes,'counts':counts,'unknown':unknown,'large_source_ratio_review_rows':large,'new_GETs':0,'Qlib_calls':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,'ready_for_independent_source_review':True,'ready_for_Qlib_map':False};(card/'numeric_outlier_repair_result.json').write_text(json.dumps(result,indent=2)+'\n');print({'status':result['status'],'counts':counts,'changes':changes,'unknown':unknown})


{'status': 'source_numeric_124_unambiguous_and_two_AMD_qualitative_unknown_no_market_outcomes', 'counts': {'AMD': {'numeric': 61, 'qualitative_unknown': 2}, 'NVDA': {'numeric': 63, 'qualitative_unknown': 0}}, 'changes': [{'accession': '0000002488-25-000106', 'filing_date': '2025-08-05', 'old_reported_USD_million': 3200.0, 'new_reported_USD_million': 7700.0, 'unchanged_forward_mid_USD_million': 8700.0, 'new_growth': 0.1298701298701299, 'original_quarter_sentence': 'Second quarter revenue was a record $7.7 billion'}, {'accession': '0000002488-25-000009', 'filing_date': '2025-02-04', 'old_reported_USD_million': 25800.0, 'new_reported_USD_million': 7700.0, 'unchanged_forward_mid_USD_million': 7100.0, 'new_growth': -0.07792207792207795, 'original_quarter_sentence': 'Fourth quarter revenue was a record $7.7 billion'}, {'accession': '0000002488-26-000014', 'filing_date': '2026-02-03', 'old_reported_USD_million': 34600.0, 'new_reported_USD_million': 10300.0, 'unchanged_forward_mid_USD_million'

## 中文结论

第三轮仅修三份 AMD 原始公告明确“本季收入创纪录”句；最终 124 数值、两期 AMD 定性未知，独立复核通过来源口径。 见 `conclusion.md`。